# Injecting freshwater into a saltwater aquifer: two-fluid SWI and variable density (BUY)

The companion notebooks [`mf6-swi-buy-a`](mf6-swi-buy-a.ipynb) and [`mf6-swi-buy-b`](mf6-swi-buy-b.ipynb) let freshwater flow toward the sea over saltwater. This notebook turns the problem around: a well **injects freshwater into a confined aquifer that holds only saltwater**, as in aquifer storage and recovery (ASR), where freshwater is stored in a brackish or saline aquifer and pumped back out later.

The well injects for a year, the freshwater is stored for a year, and the same well then pumps it back. The freshwater floats on the saltwater: it forms a lens beneath the top of the aquifer, the lens spreads while it is stored, and saltwater rises toward the well when it pumps. SWI runs in **two-fluid** mode, so the saltwater moves out of the way of the freshwater and back again. A variable-density model of the same aquifer checks the result.

> **Preview package.** SWI is not yet part of a MODFLOW 6 release. The class MODFLOW 6 includes it; see the [README](../../README.md).

## What this notebook covers

By the end of this notebook you will be able to:

- start a two-fluid SWI model with an aquifer that holds only saltwater,
- give the freshwater model a boundary that lets freshwater leave to the sea but brings none in,
- follow a freshwater lens through injection, storage, and recovery, and let SWI cut back the well when saltwater reaches it,
- compare the SWI lens with the transition zone of a variable-density model, and what each model says about how much of the freshwater comes back.

## The problem

A confined aquifer 30 m thick lies beneath a confining unit, from 30 to 60 m below sea level. The section runs 2 km, and both ends are open to the sea, so the aquifer is full of seawater. The confining unit is not part of the model: the top of the aquifer has no flow. The section is 1 m wide, so the well is a row of wells, or a gallery, and every flow is per meter of that row.

| Property | Value |
|---|---|
| Length | 2,000 m; well at the center |
| Well screen | top 10 m of the aquifer, -30 to -40 m |
| Top and bottom of the aquifer | -30 m and -60 m |
| Hydraulic conductivity, $K$ | 10 m/d |
| Porosity | 0.2 |
| Specific storage | 1 × 10⁻⁵ 1/m |
| Relative density difference, $\delta$ | 0.025 (seawater 1,025 kg/m³) |
| Salt concentration of seawater | 35 kg/m³ |
| Injection, year 1 | 1.5 m³/d per meter of well row |
| Storage, year 2 | no pumping |
| Recovery, year 3 | 1.5 m³/d per meter of well row, cut back when saltwater reaches the well |

A year of injection stores 548 m³ of freshwater per meter of well row. If it filled the whole thickness of the aquifer around the well, it would make a block of freshwater $548 / (0.2 \times 30) = 91$ m wide, reaching 46 m to each side. Because it floats, it spreads much farther along the top of the aquifer.

## Imports and settings

In [ ]:
%matplotlib inline
import io
import pathlib as pl
import time

import flopy
import ipywidgets as widgets
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

width = 17.15 / 2.54  # figure width, inches

All lengths are in meters and times in days. The grid has 200 columns 10 m wide; the well is in column 100, 1,000 m from the western end. Each of the three periods lasts a year, in weekly time steps.

In [ ]:
length, delr = 2000.0, 10.0  # m
ncol = int(length / delr)
x = (np.arange(ncol) + 0.5) * delr  # cell centers, from the western end
x_edges = np.arange(ncol + 1) * delr
top, bot = -30.0, -60.0  # m
thick = top - bot
k, porosity, ss = 10.0, 0.2, 1.0e-5
delta, c_sea = 0.025, 35.0
drhodc = 1000.0 * delta / c_sea  # seawater 1,025 kg/m3
q = 1.5  # m3/d per meter of well row
jw = ncol // 2  # well column
x_well = x[jw]

year = 365.25
periods = ["injection", "storage", "recovery"]
rates = [q, 0.0, -q]  # well rate in each period; injection is positive
nstp = 52  # weekly time steps

## The two-fluid SWI model

Two groundwater flow models share the one-layer grid, coupled by the SWI-SWI exchange. The **saltwater model** has a hydraulic conductivity $1+\delta$ times that of the freshwater model, and the sea at both ends: a General-Head Boundary (**GHB**) holds the saltwater head at 0 m, with the conductance of the half cell between the cell center and the end of the aquifer.

The **freshwater model** has the well, with the Well Package `AUTO_FLOW_REDUCE` option, which cuts back the well when the freshwater zone of its cell becomes thinner than a tenth of the cell. That only matters when the well pumps: during injection the well puts in its full rate.

**The freshwater model also needs the sea.** Where a cell holds only saltwater, its freshwater head is the head that puts the interface at the top of the cell, $h_f = -\delta\, z_\mathrm{top} + (1+\delta)\, h_s$, which is 0.75 m where the saltwater head is at sea level. A Drain (**DRN**) at each end, at that head of 0.75 m and with the same conductance as the GHB, lets freshwater that reaches the end flow out to the sea. A drain, not a GHB, because the sea holds no freshwater: when the well pumps and the heads fall, a GHB would let freshwater flow in from the sea, and a drain does not.

**Start with saltwater only.** The saltwater head starts at 0 m and the freshwater head at 0.75 m, so the interface starts at the top of the aquifer, and every cell holds only saltwater. Start the freshwater head there, not lower. A freshwater head of 0 m also means that every cell holds only saltwater, with the interface 30 m above the top of the aquifer, but from there the first injection step does not converge.

As in the other SWI notebooks, the model uses the Newton formulation, a specific yield in the Storage (**STO**) Package, and backtracking in the Iterative Model Solution (**IMS**).

In [ ]:
swi_ws = pl.Path("models/mf6-swi-buy-c/swi-2fluid")
sim = flopy.mf6.MFSimulation(sim_ws=swi_ws, exe_name="mf6")
flopy.mf6.ModflowTdis(
    sim, nper=len(periods), perioddata=[(year, nstp, 1.0)] * len(periods)
)
h_top = -delta * top  # freshwater head with the interface at the top, 0.75 m
for name in ("fresh", "salt"):
    f = 1.0 + delta if name == "salt" else 1.0
    gwf = flopy.mf6.ModflowGwf(sim, modelname=name, newtonoptions="newton")
    flopy.mf6.ModflowGwfdis(
        gwf, nlay=1, nrow=1, ncol=ncol, delr=delr, delc=1.0, top=top, botm=bot
    )
    flopy.mf6.ModflowGwfic(gwf, strt=h_top if name == "fresh" else 0.0)
    flopy.mf6.ModflowGwfnpf(gwf, icelltype=0, k=f * k)
    flopy.mf6.ModflowGwfsto(gwf, iconvert=0, sy=porosity, ss=ss, transient={0: True})
    cond = f * k * thick / (0.5 * delr)  # half cell at each end
    ends = [(0, 0, 0), (0, 0, ncol - 1)]
    if name == "fresh":
        flopy.mf6.ModflowGwfdrn(
            gwf, pname="sea", stress_period_data={0: [(c, h_top, cond) for c in ends]}
        )
        wel = {i: [((0, 0, jw), r)] for i, r in enumerate(rates) if r != 0.0}
        wel[1] = []  # storage: the well is off
        flopy.mf6.ModflowGwfwel(
            gwf,
            pname="well",
            stress_period_data=wel,
            save_flows=True,
            auto_flow_reduce=0.1,
            afrcsv_filerecord="fresh.afr.csv",
        )
    else:
        flopy.mf6.ModflowGwfghb(
            gwf, pname="sea", stress_period_data={0: [(c, 0.0, cond) for c in ends]}
        )
    flopy.mf6.ModflowGwfswi(
        gwf, relative_density_difference=delta, zeta_filerecord=f"{name}.zta"
    )
    flopy.mf6.ModflowGwfoc(
        gwf,
        head_filerecord=f"{name}.hds",
        budget_filerecord=f"{name}.cbc",
        saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
    )

flopy.mf6.ModflowSwiswi(sim, exgtype="SWI6-SWI6", exgmnamea="fresh", exgmnameb="salt")
ims = flopy.mf6.ModflowIms(
    sim,
    outer_dvclose=1.0e-4,
    outer_maximum=500,
    inner_maximum=100,
    inner_dvclose=1.0e-5,
    linear_acceleration="bicgstab",
    backtracking_number=20,
    backtracking_tolerance=1.05,
    backtracking_reduction_factor=0.1,
    backtracking_residual_limit=0.002,
)
sim.register_ims_package(ims, ["fresh", "salt"])
sim.write_simulation(silent=True)
start = time.perf_counter()
success, buff = sim.run_simulation(silent=True)
assert success, "MODFLOW 6 did not terminate normally"
print(f"SWI run time {time.perf_counter() - start:.1f} s")

Read the interface and the well rates. `fresh_volume()` gives the volume of freshwater in the aquifer, per meter of well row: the porosity times the thickness of the freshwater zone, summed over the cells. `lens_edges()` gives the western and eastern edges of the lens, the outermost cells with more than 0.1 m of freshwater. The well budget gives the rate the well actually pumped in each time step, after any cutback.

In [ ]:
zeta_file = flopy.utils.HeadFile(swi_ws / "fresh.zta", text="zeta")
swi_years = np.array(zeta_file.get_times()) / year
swi_zeta = zeta_file.get_alldata()[:, 0, 0, :]
fresh_thick = top - np.clip(swi_zeta, bot, top)  # freshwater zone in each cell


def fresh_volume(b):
    return porosity * b.sum() * delr


def lens_edges(b, min_thick=0.1):
    fresh = x[b > min_thick]
    return (fresh.min(), fresh.max()) if fresh.size else (np.nan, np.nan)


cbc = flopy.utils.CellBudgetFile(swi_ws / "fresh.cbc")
well_rate = np.zeros(swi_years.size)  # actual rate, injection positive
for kstp, kper in cbc.get_kstpkper():
    rec = cbc.get_data(kstpkper=(kstp, kper), text="WEL")
    if rec:
        well_rate[kper * nstp + kstp] = rec[0]["q"].sum()
dt = year / nstp
recovery = swi_years > 2.0
recovered = -(well_rate[recovery] * dt).sum()
injected = (well_rate[swi_years <= 1.0] * dt).sum()

for i, p in enumerate(periods):
    n = (i + 1) * nstp - 1  # last time step of the period
    w, e = lens_edges(fresh_thick[n])
    print(
        f"end of {p:9s}: {fresh_volume(fresh_thick[n]):5.0f} m3 of freshwater, "
        f"lens from {w:4.0f} to {e:4.0f} m ({e - w + delr:3.0f} m wide), "
        f"{fresh_thick[n, jw]:4.1f} m thick at the well"
    )
afr = pd.read_csv(swi_ws / "fresh.afr.csv")
first_cut = afr["time"].min() / year - 2.0
print(
    f"\ninjected {injected:.0f} m3; recovered {recovered:.0f} m3, "
    f"{100 * recovered / injected:.0f} percent"
)
print(
    f"well cut back from {12 * first_cut:.1f} months into recovery; "
    f"pumping {100 * -well_rate[-1] / q:.0f} percent of its rate at the end"
)

**What to look for.** During injection the freshwater pushes the interface down to 48.9 m below sea level at the well, almost two thirds of the way through the aquifer, and spreads along the top of the aquifer into a lens 350 m wide, almost four times as wide as a block of freshwater filling the whole thickness. During storage nothing is added or removed, but the lens keeps spreading: the freshwater flows outward along the top of the aquifer, the saltwater flows back in beneath it, and the lens becomes wider (410 m) and thinner, from 18.9 m to 11.0 m at the well. When the well pumps, saltwater rises beneath it in a cone. The freshwater zone of the well cell falls below 3 m, a tenth of the cell, and SWI starts cutting back the well 1.8 months into recovery. The well keeps pumping freshwater from the lens around it at a falling rate, 14 percent of its full rate at the end of the year. It recovers 216 m³, 39 percent of what was injected; the rest, 331 m³, is still in the aquifer, in a ring around the well. None of the freshwater reaches the ends of the aquifer. The two-fluid model runs in less than a second.

Plot the lens at the end of each period. Freshwater is blue, saltwater orange.

In [ ]:
fresh_color, salt_color = "#9ecae1", "#fdae6b"
ends_of = [(i + 1) * nstp - 1 for i in range(len(periods))]
xlim = (x_well - 400.0, x_well + 400.0)

with flopy.plot.styles.USGSPlot():
    fig, axs = plt.subplots(
        3, 1, figsize=(width, 0.75 * width), sharex=True, layout="constrained"
    )
    for ax, p, n in zip(axs, periods, ends_of):
        ax.fill_between(x, bot, top, color=salt_color, lw=0)
        ax.fill_between(
            x, np.maximum(swi_zeta[n], bot), top, color=fresh_color, lw=0, step="mid"
        )
        ax.step(x, np.clip(swi_zeta[n], bot, top), where="mid", color="black", lw=1.0)
        ax.axvline(x_well, color="0.3", lw=0.75, ls=":")
        ax.set_ylim(bot, top)
        ax.set_ylabel("Elevation (m)")
        flopy.plot.styles.heading(ax=ax, heading=f"End of {p}, year {n // nstp + 1}")
    axs[-1].set_xlim(*xlim)
    axs[-1].set_xlabel("Distance from the western end (m)")
    axs[0].text(x_well + 5, -57, "well", fontsize=8)
    plt.show()

## The variable-density model

The same aquifer in 30 layers 1 m thick. The flow model gets the Buoyancy (**BUY**) Package, and a transport model carries the salt, with a longitudinal dispersivity of 1 m, a transverse dispersivity of 0.1 m, and the TVD advection scheme, as in the other two notebooks.

- The aquifer starts full of seawater, at rest: the concentration is 35 kg/m³ everywhere, and the freshwater head of each layer is that of seawater standing at sea level, $h = -\delta\, z$.
- The sea, a GHB with a head of 0 m and the concentration of seawater, lies on both ends of every layer. BUY gives it the density of seawater, so the pressure of the sea grows with depth. Water that leaves through it carries the concentration of the aquifer, and water that comes in carries seawater.
- The well is screened in the top 10 m of the aquifer, with its rate split equally among those 10 layers. It injects water with a concentration of 0. The variable-density model has no `AUTO_FLOW_REDUCE`: during recovery the well pumps its full rate, whatever the water it draws.

The transport solution uses delta-bar-delta under-relaxation (`under_relaxation="dbd"`). Without it the TVD scheme flips back and forth between two answers in the cells next to the well, where freshwater enters an aquifer full of seawater, and the first time steps do not converge.

The salt starts from a known state, an aquifer full of seawater, so this model needs no period to settle before the well starts, unlike the model in [`mf6-swi-buy-b`](mf6-swi-buy-b.ipynb).

In [ ]:
nlay, delz = 30, 1.0
botm = top - delz * (np.arange(nlay) + 1.0)
zc = top - delz * (np.arange(nlay) + 0.5)  # cell centers
alpha_l, alpha_t = 1.0, 0.1  # m

buy_ws = pl.Path("models/mf6-swi-buy-c/buy")
bsim = flopy.mf6.MFSimulation(sim_ws=buy_ws, exe_name="mf6")
flopy.mf6.ModflowTdis(
    bsim, nper=len(periods), perioddata=[(year, nstp, 1.0)] * len(periods)
)
flow = flopy.mf6.ModflowGwf(bsim, modelname="flow")
ims = flopy.mf6.ModflowIms(
    bsim,
    outer_dvclose=1.0e-5,
    outer_maximum=50,
    inner_maximum=200,
    inner_dvclose=1.0e-7,
    linear_acceleration="bicgstab",
    filename="flow.ims",
)
bsim.register_ims_package(ims, ["flow"])
dis_kw = dict(nlay=nlay, nrow=1, ncol=ncol, delr=delr, delc=1.0, top=top, botm=botm)
flopy.mf6.ModflowGwfdis(flow, **dis_kw)
strt = np.repeat(-delta * zc, ncol).reshape(nlay, 1, ncol)  # seawater at rest
flopy.mf6.ModflowGwfic(flow, strt=strt)
flopy.mf6.ModflowGwfnpf(flow, icelltype=0, k=k)
flopy.mf6.ModflowGwfsto(flow, iconvert=0, ss=ss, transient={0: True})
flopy.mf6.ModflowGwfbuy(
    flow, packagedata=[(0, drhodc, 0.0, "transport", "concentration")]
)
cond = k * delz / (0.5 * delr)
sea = [((kk, 0, j), 0.0, cond, c_sea) for kk in range(nlay) for j in (0, ncol - 1)]
flopy.mf6.ModflowGwfghb(
    flow, auxiliary="concentration", pname="sea", stress_period_data={0: sea}
)
screen = range(10)  # top 10 layers, -30 to -40 m
wel = {
    i: [((kk, 0, jw), r / len(screen), 0.0) for kk in screen]
    for i, r in enumerate(rates)
    if r != 0.0
}
wel[1] = []
flopy.mf6.ModflowGwfwel(
    flow, auxiliary="concentration", pname="well", stress_period_data=wel
)
flopy.mf6.ModflowGwfoc(
    flow, budget_filerecord="flow.cbc", saverecord=[("BUDGET", "ALL")]
)

transport = flopy.mf6.ModflowGwt(bsim, modelname="transport")
imt = flopy.mf6.ModflowIms(
    bsim,
    outer_dvclose=1.0e-4,
    outer_maximum=100,
    inner_maximum=300,
    inner_dvclose=1.0e-6,
    linear_acceleration="bicgstab",
    # damp the TVD scheme's flip-flop next to the well
    under_relaxation="dbd",
    under_relaxation_theta=0.7,
    under_relaxation_kappa=0.1,
    under_relaxation_gamma=0.0,
    under_relaxation_momentum=0.0,
    filename="transport.ims",
)
bsim.register_ims_package(imt, ["transport"])
flopy.mf6.ModflowGwtdis(transport, **dis_kw)
flopy.mf6.ModflowGwtic(transport, strt=c_sea)
flopy.mf6.ModflowGwtmst(transport, porosity=porosity)
flopy.mf6.ModflowGwtadv(transport, scheme="tvd")
flopy.mf6.ModflowGwtdsp(transport, alh=alpha_l, ath1=alpha_t, diffc=8.6e-5)
flopy.mf6.ModflowGwtssm(
    transport,
    sources=[("sea", "aux", "concentration"), ("well", "aux", "concentration")],
)
flopy.mf6.ModflowGwtoc(
    transport,
    concentration_filerecord="transport.ucn",
    saverecord=[("CONCENTRATION", "ALL")],
)
flopy.mf6.ModflowGwfgwt(bsim, exgmnamea="flow", exgmnameb="transport")
bsim.write_simulation(silent=True)
start = time.perf_counter()
success, buff = bsim.run_simulation(silent=True)
assert success, "MODFLOW 6 did not terminate normally"
print(f"BUY run time {time.perf_counter() - start:.1f} s")

Read the concentrations as a fraction of seawater, 0 for freshwater and 1 for seawater. The freshwater content of the aquifer is the porosity times one minus that fraction, summed over the cells. The water the well pumps is a mix of its 10 layers, which all pump the same rate, so its fraction of seawater is the mean of those cells; the freshwater it recovers is the pumped volume times one minus that fraction.

In [ ]:
ucn = flopy.utils.HeadFile(buy_ws / "transport.ucn", text="concentration")
buy_years = np.array(ucn.get_times()) / year
conc = ucn.get_alldata()[:, :, 0, :] / c_sea  # fraction of seawater
buy_fresh = porosity * ((1.0 - conc) * delz * delr).sum(axis=(1, 2))
pumped_salt = conc[:, :10, jw].mean(axis=1)  # fraction of seawater, well water
buy_recovered = (q * dt * (1.0 - pumped_salt[recovery])).sum()


def buy_edges(c, level=0.5):
    # outermost columns where the top layer is fresher than level
    fresh = x[c[0] < level]
    return (fresh.min(), fresh.max()) if fresh.size else (np.nan, np.nan)


for i, p in enumerate(periods):
    n = ends_of[i]
    w, e = buy_edges(conc[n])
    print(
        f"end of {p:9s}: {buy_fresh[n]:5.0f} m3 of freshwater, 50 percent line "
        f"at the top from {w:4.0f} to {e:4.0f} m ({e - w + delr:3.0f} m wide)"
    )
for months in (1, 3, 6, 12):
    n = 2 * nstp + int(round(months * nstp / 12)) - 1
    print(
        f"{months:2d} months into recovery: well water {100 * pumped_salt[n]:3.0f} percent seawater"
    )
print(
    f"\nfreshwater recovered {buy_recovered:.0f} m3, "
    f"{100 * buy_recovered / injected:.0f} percent of what was injected"
)

**What to look for.** **What to look for.** No salt leaves the lens through the ends of the aquifer: the freshwater content is the 548 m³ that was injected at the end of injection and at the end of storage. But it is mixed. At the top of the aquifer the 50 percent seawater line reaches 210 m across at the end of injection and 290 m at the end of storage, narrower than the SWI lens, 350 and 410 m. At the start of recovery the water at the bottom of the well screen is already brackish, so the well pumps water with 4 percent seawater from its first time step, 20 percent after a month, and 67 percent at the end of the year. Over the year it pumps 288 m³ of freshwater, 52 percent of what was injected, mixed with about as much seawater. The variable-density model takes about 8 seconds, 20 times as long as SWI.

## Compare the two models

Draw the fraction of seawater in the variable-density model at the end of each period, with contours at 10, 50, and 90 percent seawater, and the SWI interface on top.

In [ ]:
cmap = mpl.colors.LinearSegmentedColormap.from_list("fs", [fresh_color, salt_color])
z_edges = np.r_[top, botm]

with flopy.plot.styles.USGSPlot():
    fig, axs = plt.subplots(
        3, 1, figsize=(width, 0.8 * width), sharex=True, layout="constrained"
    )
    for ax, p, n in zip(axs, periods, ends_of):
        pc = ax.pcolormesh(x_edges, z_edges, conc[n], cmap=cmap, vmin=0.0, vmax=1.0)
        cs = ax.contour(
            x, zc, conc[n], levels=[0.1, 0.5, 0.9], colors="0.25", linewidths=0.75
        )
        ax.clabel(cs, fmt=lambda v: f"{v:.0%}", fontsize=7)
        lens = np.where(fresh_thick[n] > 1.0e-3, np.clip(swi_zeta[n], bot, top), np.nan)
        ax.step(x, lens, where="mid", color="black", lw=1.5, label="SWI interface")
        ax.axvline(x_well, color="0.3", lw=0.75, ls=":")
        ax.set_ylim(bot, top)
        ax.set_ylabel("Elevation (m)")
        flopy.plot.styles.heading(ax=ax, heading=f"End of {p}, year {n // nstp + 1}")
    axs[-1].set_xlim(*xlim)
    axs[-1].set_xlabel("Distance from the western end (m)")
    flopy.plot.styles.graph_legend(ax=axs[0], loc="lower left")
    fig.colorbar(pc, ax=axs, label="Fraction of seawater", shrink=0.6)
    plt.show()

**What to look for.** Beneath the well, the SWI interface follows the 50 percent seawater line through all three periods: at the end of injection SWI puts the interface at -48.9 m and the variable-density model puts the 50 percent line at -50.5 m; at the end of storage, -41.0 m and -43.5 m. The models part at the edges of the lens, where SWI has a thin layer of freshwater, a meter or two thick, beneath the top of the aquifer. In the variable-density model, mixing turns that thin layer into brackish water, so its 50 percent line reaches less far. At the end of storage the transition zone at the well is 11 m thick, between the 10 and 90 percent lines, as thick as the freshwater above it.

Both models agree that the freshwater **floats**: injected through a screen in the top 10 m, it spreads along the top of the aquifer, and beneath the well the saltwater rises back as soon as the well stops injecting.

## Watch the lens grow, spread, and shrink

Step through the three years with the slider, or press the play button. Each frame is two weeks. The upper panel is SWI, the lower panel the variable-density model with the SWI interface on top; the triangle in the SWI panel marks the well, which has no depth there: it injects into and pumps from the freshwater zone of its one cell, wherever the interface lies. The gray bar in the lower panel is the screen of the variable-density well, in the top 10 m. The heading gives the rate the well pumps in each model. The dashed line in the lower panel is the SWI interface at the end of injection, for reference.

The frames are drawn once, before the controls appear, so they play smoothly; that takes a few seconds.

In [ ]:
from IPython.display import display

frames = np.r_[np.arange(0, 3 * nstp, 2), 3 * nstp - 1]  # every two weeks
z_inject = np.where(fresh_thick[nstp - 1] > 1.0e-3, swi_zeta[nstp - 1], np.nan)


def well_text(n):
    p = n // nstp
    if p == 0:
        return f"injecting {q:.1f} m³/d in both models"
    if p == 1:
        return "well off"
    return (
        f"pumping: SWI {-well_rate[n]:.2f} m³/d, variable density {q:.1f} m³/d "
        f"with {100 * pumped_salt[n]:.0f} percent seawater"
    )


def draw_frame(n):
    with flopy.plot.styles.USGSPlot():
        fig, (ax0, ax1) = plt.subplots(
            2, 1, figsize=(width, 0.62 * width), sharex=True, layout="constrained"
        )
        ax0.fill_between(x, bot, top, color=salt_color, lw=0)
        ax0.fill_between(
            x, np.maximum(swi_zeta[n], bot), top, color=fresh_color, lw=0, step="mid"
        )
        ax0.step(x, np.clip(swi_zeta[n], bot, top), where="mid", color="black", lw=1.0)
        ax1.pcolormesh(x_edges, z_edges, conc[n], cmap=cmap, vmin=0.0, vmax=1.0)
        if conc[n].min() < 0.5:
            ax1.contour(x, zc, conc[n], levels=[0.5], colors="0.25", linewidths=0.75)
        lens = np.where(fresh_thick[n] > 1.0e-3, np.clip(swi_zeta[n], bot, top), np.nan)
        ax1.step(x, lens, where="mid", color="black", lw=1.5)
        ax1.step(x, z_inject, where="mid", color="black", lw=0.75, ls="--")
        for ax, title in (
            (ax0, "SWI"),
            (ax1, "Variable density, 50 percent line, and SWI interface"),
        ):
            ax.set_ylim(bot, top)
            ax.set_ylabel("Elevation (m)")
            flopy.plot.styles.heading(ax=ax, heading=title)
        # SWI: the well is a source in the one cell, with no screen depth
        ax0.plot(x_well, top, marker="v", color="0.3", ms=8, clip_on=False)
        # variable density: the screen in the top 10 layers
        ax1.plot([x_well, x_well], [top, top - 10.0], color="0.3", lw=3)
        ax1.set_xlim(*xlim)
        ax1.set_xlabel("Distance from the western end (m)")
        fig.suptitle(
            f"{swi_years[n]:4.2f} years, {periods[n // nstp]}: {well_text(n)}",
            fontsize=9,
        )
        buffer = io.BytesIO()
        fig.savefig(buffer, format="png", dpi=90)
        plt.close(fig)
    return buffer.getvalue()


start = time.perf_counter()
pictures = [draw_frame(n) for n in frames]
print(f"{len(frames)} frames drawn in {time.perf_counter() - start:.0f} s")

picture = widgets.Image(value=pictures[0], format="png")
play = widgets.Play(min=0, max=len(frames) - 1, step=1, interval=150)
slider = widgets.IntSlider(
    min=0, max=len(frames) - 1, readout=False, layout=widgets.Layout(width="500px")
)
widgets.jslink((play, "value"), (slider, "value"))
slider.observe(
    lambda change: setattr(picture, "value", pictures[change["new"]]), "value"
)
display(widgets.VBox([widgets.HBox([play, slider]), picture]))

**What to look for.** During injection the freshwater rises from the well screen and spreads along the top of the aquifer in both models; the variable-density lens is fresh in its core and brackish at its edges. Once the well stops, the saltwater beneath it rises steadily, from -48.9 m to -41.0 m over the year in SWI, and the lens flattens and widens; the dashed line shows how far it has come from the end of injection. When the well pumps, the saltwater rises in a cone beneath it, the SWI rate falls as the freshwater zone of the well cell thins, and the variable-density well draws more and more seawater, leaving a ring of freshwater around the well in both models.

## The freshwater in the aquifer, and the water the well pumps

The upper panel shows the volume of freshwater in the aquifer in both models: in SWI the freshwater above the interface, in the variable-density model the freshwater content of the mixed water. The lower panel shows what the well pumps during recovery: the rate SWI lets it pump, as a fraction of its full rate, and the fraction of seawater in the water the variable-density well pumps at its full rate.

In [ ]:
swi_fresh = np.array([fresh_volume(b) for b in fresh_thick])

with flopy.plot.styles.USGSPlot():
    fig, (ax0, ax1) = plt.subplots(
        2, 1, figsize=(width, 0.7 * width), sharex=True, layout="constrained"
    )
    ax0.plot(swi_years, swi_fresh, color="black", label="SWI")
    ax0.plot(buy_years, buy_fresh, color="C0", label="Variable density")
    ax0.set_ylabel("Freshwater in the aquifer\n(m³ per meter of well row)")
    flopy.plot.styles.graph_legend(ax=ax0, loc="lower right")
    flopy.plot.styles.heading(ax=ax0, heading="Freshwater stored")
    for ax in (ax0, ax1):
        for t in (1.0, 2.0):
            ax.axvline(t, color="0.6", lw=0.75, ls="--")
    ax1.plot(
        np.r_[2.0, swi_years[recovery]],
        np.r_[1.0, -well_rate[recovery] / q],
        color="black",
        label="SWI, rate pumped as a fraction of the full rate",
    )
    ax1.plot(
        np.r_[2.0, buy_years[recovery]],
        np.r_[pumped_salt[2 * nstp - 1], pumped_salt[recovery]],
        color="C1",
        label="Variable density, fraction of seawater pumped",
    )
    ax1.set_ylim(0.0, 1.05)
    ax1.set_xlim(0.0, 3.0)
    ax1.set_xlabel("Time (years)")
    ax1.set_ylabel("Fraction")
    flopy.plot.styles.graph_legend(ax=ax1, loc="upper left")
    flopy.plot.styles.heading(ax=ax1, heading="Recovery")
    for t, p in zip((0.5, 1.5, 2.5), periods):
        ax0.text(t, 0.97, p, transform=ax0.get_xaxis_transform(), ha="center", va="top")
    plt.show()

**What to look for.** In both models the freshwater in the aquifer rises at the injection rate and holds steady during storage. During recovery the two models answer different questions. SWI pumps only freshwater: it cuts back the well when the saltwater rises into the well cell, and the rate falls steeply in the second and third months. The variable-density well pumps its full rate throughout, and the seawater in the water it pumps rises from 4 percent at once to 67 percent. Neither recovers much more than half the freshwater. The lens spreads during storage, and the saltwater rises beneath the well when it pumps: buoyancy costs most of the freshwater, whatever the model.

### Try it

- **Store for longer, or not at all.** Change the storage period in `TDIS` (both models), or skip it. How fast does the lens spread when the well is off, and how much does that cost at recovery?
- **Inject faster.** Double `q` and halve the injection time (both models). Is the lens any thicker at the well? Does more of the freshwater come back?
- **A thicker aquifer.** Make the aquifer 60 m thick (`bot = -90.0`, and `nlay = 60` in the variable-density model). Freshwater floats on top whatever the thickness, so does a thicker aquifer help or hurt?
- **More layers in SWI.** Give the SWI model 3 layers 10 m thick, with the well in the top layer only. The lens hardly changes, within half a meter: SWI places the interface within a cell, so one layer is enough for an aquifer, even with freshwater injected into it.

## Recap

You built a confined aquifer that holds only seawater, injected freshwater into it, stored it, and pumped it back. **Two-fluid SWI handles freshwater injected where there is none.** Two things make it work: the freshwater model starts with the interface at the top of the aquifer, and it has a way to send freshwater to the sea but not to take it in, here a drain at each end at the freshwater head of the sea at the top of the aquifer.

The SWI lens matches the 50 percent seawater line of a variable-density model beneath the well and spreads at the same pace during storage, in one layer and in a twentieth of the time. Both models show why storing freshwater in a saltwater aquifer is hard: the freshwater floats and spreads along the top of the aquifer, and saltwater rises beneath the well when it pumps. SWI tells how much freshwater is left and when the well has to be cut back; the variable-density model also tells how salty the well water gets, because the transition zone, not the 50 percent line, reaches the well screen first.